# PSP8 — Coleta e ETL em nuvem (Kaggle → Google Colab → Google Drive)

**Etapa 2:** validar persistência de um modelo de dados *flat*, com camadas bronze (arquivo de origem preservado), prata (atributos tipados) e evidências de qualidade e integridade. Este notebook **não altera** o notebook original do SSD.

A base é de projetos de infraestrutura em geral, não de data centers. Este fluxo ainda **não** executa análises de viabilidade nem comprova desempenho em situações reais.


## 1. Preparar o Colab
1. No Colab, execute todas as células em sequência, com acesso à internet.
2. Autorize a montagem do Google Drive quando solicitado.
3. Depois da execução, confira os CSVs nas pastas `bronze` e `prata`, a auditoria, o manifesto e os hashes.
4. Guarde capturas reais da pasta do Drive e das mensagens de validação para as evidências acadêmicas. **Não** publique credenciais nem arquivos privados.


In [ ]:
%pip install -q kagglehub pandas


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Carregar o código versionado
O Colab baixa a implementação do pipeline a partir da branch de trabalho. Para uma entrega reprodutível final, a URL deve ser fixada em um commit validado antes de mesclar na `main`.


In [ ]:
from pathlib import Path
import urllib.request

URL_SCRIPT = (
    'https://raw.githubusercontent.com/ricardopimentelandre/'
    'MVP-Viabilidade-Nova-Unidade-Dados/psp8-etapa-2-pipeline/'
    'scripts/pipeline_etl_drive.py'
)
arquivo_script = Path('/content/pipeline_etl_drive.py')
urllib.request.urlretrieve(URL_SCRIPT, arquivo_script)
print('Código de ETL obtido:', arquivo_script)


In [ ]:
import sys
sys.path.insert(0, '/content')
from pipeline_etl_drive import executar_pipeline

DESTINO_DRIVE = Path('/content/drive/MyDrive/UnB/PSP8/MVP_Viabilidade_Nova_Unidade')
resultado = executar_pipeline(DESTINO_DRIVE)


## 3. Validar os arquivos persistidos
O pipeline já realizou a gravação **e releitura do Drive** e registrou os hashes. As células abaixo exibem o manifesto de execução e o resumo da auditoria. Não invente execução: se esta etapa falhar, registre a falha e corrija antes de afirmar persistência.


In [ ]:
import json
from IPython.display import display

manifesto = resultado['manifesto']
auditoria = resultado['auditoria']
print('Estado:', manifesto['status'])
print('Local no Drive:', manifesto['destino_execucao'])
print('Registros bronze/prata:', manifesto['registros_bronze'], manifesto['registros_prata'])
print('SHA-256 bronze:', manifesto['sha256_bronze'])
print('SHA-256 prata:', manifesto['sha256_prata'])
print('Releitura:', manifesto['releitura_bronze_e_prata'])
print('Linhas duplicadas:', auditoria['linhas_duplicadas_exatas'])
print('IDs duplicados:', auditoria['project_id_duplicados'])


In [ ]:
import pandas as pd
prata_do_drive = pd.read_csv(resultado['caminho_prata'])
assert len(prata_do_drive) == manifesto['registros_prata']
assert len(prata_do_drive.columns) == 13
display(prata_do_drive.head(3))
resumo_qualidade = pd.DataFrame(auditoria['atributos']).T
display(resumo_qualidade[['tipo_pandas', 'nulos', 'valores_unicos']])


## 4. Limitações e próximo ponto de controle
- O CSV original é preservado em `bronze`; a camada `prata` preserva as 13 colunas e tipa os campos numéricos. Não há exclusão de registros nesta etapa.
- A coleta e a licença devem ser verificadas na página da fonte; o manifesto marca a licença como **pendente** até confirmação.
- A auditoria identifica indícios de problemas, mas não corrige nem garante acurácia semântica.
- Os dados **não** devem ser considerados persistidos no Drive até este notebook executar com sucesso e a pasta ser verificada.
- A análise das perguntas P1–P4 e o short paper são etapas posteriores, sujeitas à validação.
